In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


In [3]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title from state
    title = state['title']

    # call llm for gen outline
    prompt = f"Create a detailed outline for a blog post with the title: {title}"
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    # fetch title and outline from state
    title = state['title']
    outline = state['outline']

    # call llm for gen blog content
    prompt = f"Write a detailed blog post with the title: {title} and the following outline: \n{outline}"
    content = model.invoke(prompt).content

    # update state
    state['content'] = content

    return state

In [10]:
graph = StateGraph(BlogState)

#add nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

#add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
initial_state = {'title' : 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)
print(final_state['content'])